# LeetCode 263 — Ugly Number bằng Python

**Link đề:** https://leetcode.com/problems/ugly-number/
**Độ khó:** Easy · **Chủ đề:** Math

```python
class Solution:
    def isUgly(self, n: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **thừa số nguyên tố** (prime factor) và phân tích một số ra thừa số.
2. Viết baseline "phân tích đầy đủ" rồi nhận ra ta **không cần biết hết** các thừa số.
3. Nắm lời giải chính: **chia hết cho 2, 3, 5 bao nhiêu lần có thể**, còn lại phải là 1.
4. Xử lý đúng biên: `n <= 0` (âm, 0) và `n = 1`.
5. Biết thêm 2 cách sáng tạo: mẹo bit `n & -n` và "số chia hết lớn nhất" `2^30·3^19·5^13`.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

**Ugly number** là **số nguyên dương** mà các thừa số nguyên tố của nó chỉ gồm `2`, `3`, `5`.

Cho số nguyên `n`, trả `True` nếu `n` là ugly number.

| `n` | Kết quả | Vì sao |
|---:|:---:|---|
| `6` | `True` | `6 = 2 × 3` |
| `1` | `True` | `1` không có thừa số nguyên tố nào → không vi phạm |
| `14` | `False` | `14 = 2 × 7`, có thừa số `7` |
| `0` | `False` | không phải số dương |
| `-6` | `False` | không phải số dương |

**Constraints:** `-2^31 <= n <= 2^31 - 1` → **có số âm và số 0** trong input, phải xử lý.

Câu hỏi định hướng: "chỉ gồm 2, 3, 5" nghĩa là sau khi **gỡ hết** các thừa số 2, 3, 5 ra khỏi `n` thì không còn gì (còn lại `1`).

In [1]:
# Kiểm tra bằng tay các ví dụ
print("6 = 2 * 3 ->", 2 * 3 == 6)
print("14 = 2 * 7 ->", 2 * 7 == 14, "(7 là số nguyên tố khác 2, 3, 5)")
print("30 = 2 * 3 * 5 ->", 2 * 3 * 5 == 30)

6 = 2 * 3 -> True
14 = 2 * 7 -> True (7 là số nguyên tố khác 2, 3, 5)
30 = 2 * 3 * 5 -> True


## 2. Nền tảng: số nguyên tố, thừa số, `%` và `//`

### 2.1. Số nguyên tố và phân tích thừa số

- **Số nguyên tố**: số nguyên `> 1` chỉ chia hết cho `1` và chính nó: `2, 3, 5, 7, 11, 13, ...`
- **Định lý cơ bản của số học**: mọi số nguyên `> 1` viết được **duy nhất** thành tích các số nguyên tố, ví dụ `360 = 2^3 · 3^2 · 5`.

### 2.2. Kiểm tra chia hết và chia nguyên

- `n % p == 0` ⇔ `n` chia hết cho `p`.
- `n // p` là thương nguyên. Dùng `n //= p` để "gỡ" một thừa số `p` ra khỏi `n`.

In [2]:
n = 360
print("360 % 2 =", 360 % 2, "-> chia hết cho 2")
print("360 % 7 =", 360 % 7, "-> không chia hết cho 7")
while n % 2 == 0:
    n //= 2
    print("gỡ một thừa số 2, còn", n)
print("Không còn chia hết cho 2:", n)

360 % 2 = 0 -> chia hết cho 2
360 % 7 = 3 -> không chia hết cho 7
gỡ một thừa số 2, còn 180
gỡ một thừa số 2, còn 90
gỡ một thừa số 2, còn 45
Không còn chia hết cho 2: 45


### 2.3. Vòng `while` với điều kiện kép và `for` trên một tuple

- `for p in (2, 3, 5):` duyệt lần lượt 3 số trong **tuple** `(2, 3, 5)` (tuple = dãy cố định, viết bằng ngoặc tròn).
- `while n % p == 0:` lặp **chừng nào** còn chia hết.

Kết hợp: gỡ toàn bộ thừa số 2, rồi toàn bộ thừa số 3, rồi 5.

In [3]:
n = 360
for p in (2, 3, 5):
    count = 0
    while n % p == 0:
        n //= p
        count += 1
    print(f"p = {p}: gỡ {count} lần, còn n = {n}")

p = 2: gỡ 3 lần, còn n = 45
p = 3: gỡ 2 lần, còn n = 5
p = 5: gỡ 1 lần, còn n = 1


### 2.4. Bẫy: `0 % p == 0` luôn đúng!

`0` chia hết cho mọi số, và `0 // 2 = 0`. Nếu không chặn `n == 0` trước, vòng `while n % 2 == 0` sẽ **chạy mãi mãi**. Đây là lý do số 1 để kiểm tra `n <= 0` ngay đầu hàm.

In [4]:
print("0 % 2 =", 0 % 2, "| 0 // 2 =", 0 // 2)
# Mô phỏng an toàn: chỉ chạy 5 vòng để thấy n không đổi
n = 0
for step in range(5):
    if n % 2 == 0:
        n //= 2
    print("vòng", step, "n =", n)
print("-> n luôn là 0, while sẽ không bao giờ dừng!")

0 % 2 = 0 | 0 // 2 = 0
vòng 0 n = 0
vòng 1 n = 0
vòng 2 n = 0
vòng 3 n = 0
vòng 4 n = 0
-> n luôn là 0, while sẽ không bao giờ dừng!


### 2.5. Số âm với `%` trong Python

Python cho `-6 % 2 == 0` và `-6 // 2 == -3`. Một số âm có thể "trông như" chỉ gồm thừa số 2, 3, 5, nhưng đề định nghĩa ugly number là **số dương** → luôn `False`.

In [5]:
print("-6 % 2 =", -6 % 2, "| -6 // 2 =", -6 // 2, "| -7 // 2 =", -7 // 2)

-6 % 2 = 0 | -6 // 2 = -3 | -7 // 2 = -4


## 3. Cách 1 — Baseline: phân tích đầy đủ ra thừa số nguyên tố

Ý tưởng trực giác: **tìm tất cả** thừa số nguyên tố của `n`, rồi kiểm tra chúng có nằm trong `{2, 3, 5}` không.

Thuật toán thử chia (trial division): thử `d = 2, 3, 4, ...` khi `d * d <= n`; mỗi khi chia hết thì gỡ hết `d`. Cuối cùng nếu `n > 1` thì phần còn lại là một số nguyên tố lớn.

In [6]:
def prime_factors(n: int) -> list[int]:
    factors = []
    d = 2
    while d * d <= n:
        while n % d == 0:
            factors.append(d)
            n //= d
        d += 1
    if n > 1:
        factors.append(n)        # phần còn lại là số nguyên tố
    return factors

for x in [6, 14, 360, 97, 1, 2147483647]:
    print(x, "->", prime_factors(x))

6 -> [2, 3]
14 -> [2, 7]
360 -> [2, 2, 2, 3, 3, 5]
97 -> [97]
1 -> []
2147483647 -> [2147483647]


In [7]:
def is_ugly_factorize(n: int) -> bool:
    if n <= 0:
        return False
    return set(prime_factors(n)) <= {2, 3, 5}     # <= với set: "là tập con"

print({2, 3} <= {2, 3, 5}, {2, 7} <= {2, 3, 5}, set() <= {2, 3, 5})
for x in [6, 1, 14, 0, -6, 30, 2147483647]:
    print(x, "->", is_ugly_factorize(x))

True False True
6 -> True
1 -> True
14 -> False
0 -> False
-6 -> False
30 -> True
2147483647 -> False


**`set` và phép tập con:** `set(list)` bỏ phần tử trùng; `A <= B` hỏi "mọi phần tử của `A` có nằm trong `B` không". Tập rỗng là tập con của mọi tập → `n = 1` cho `True` đúng.

**Nút thắt:** khi `n` là số nguyên tố lớn (ví dụ `2147483647`), vòng thử chia chạy tới `√n ≈ 46341` bước. Vẫn qua được, nhưng ta **phí công**: ta không cần biết thừa số lạ là `7` hay `46337` — chỉ cần biết **có** thừa số lạ hay không.

In [8]:
def count_trial_steps(n):
    steps, d = 0, 2
    while d * d <= n:
        while n % d == 0:
            n //= d
        d += 1
        steps += 1
    return steps

print("Số bước thử chia cho 2147483647:", count_trial_steps(2147483647))
print("Số bước thử chia cho 2^30:", count_trial_steps(2**30))

Số bước thử chia cho 2147483647: 46339
Số bước thử chia cho 2^30: 1


## 4. Insight — chỉ gỡ 2, 3, 5 rồi nhìn phần còn lại

Viết `n = 2^a · 3^b · 5^c · m`, trong đó `m` không chia hết cho 2, 3, 5.

- Nếu gỡ hết thừa số 2, rồi 3, rồi 5 thì còn lại đúng `m`.
- `n` là ugly ⇔ không còn thừa số nguyên tố nào khác ⇔ `m == 1`.

Vậy ta **chỉ cần tối đa** `a + b + c` phép chia, và vì `n < 2^31` nên `a + b + c <= 31`. Rất nhanh!

Ví dụ tay:

```text
n = 360: gỡ 2 -> 180 -> 90 -> 45 ; gỡ 3 -> 15 -> 5 ; gỡ 5 -> 1   => m = 1 -> True
n = 14 : gỡ 2 -> 7 ; 7 không chia hết cho 3 ; không chia hết cho 5 => m = 7 -> False
```

In [9]:
for n in [360, 14]:
    m = n
    for p in (2, 3, 5):
        while m % p == 0:
            m //= p
    print(f"n = {n}: phần còn lại m = {m} -> ugly? {m == 1}")

n = 360: phần còn lại m = 1 -> ugly? True
n = 14: phần còn lại m = 7 -> ugly? False


## 5. Lời giải chính — giảng từng đoạn rồi ghép

**Đoạn 1 — biên:**
```python
if n <= 0:
    return False        # âm và 0 không phải ugly; còn tránh vòng lặp vô hạn với 0
```

**Đoạn 2 — gỡ thừa số:**
```python
for p in (2, 3, 5):
    while n % p == 0:
        n //= p
```

**Đoạn 3 — kết luận:**
```python
return n == 1
```

Thứ tự 2, 3, 5 không quan trọng (phép chia giao hoán), nhưng **biên phải đứng đầu**.

In [10]:
def is_ugly_divide(n: int) -> bool:
    if n <= 0:
        return False
    for p in (2, 3, 5):
        while n % p == 0:
            n //= p
    return n == 1

for x in [6, 1, 14, 0, -6, 360, 7, 2147483647, 2**30]:
    print(f"{x:>11} -> {is_ugly_divide(x)}")

          6 -> True
          1 -> True
         14 -> False
          0 -> False
         -6 -> False
        360 -> True
          7 -> False
 2147483647 -> False
 1073741824 -> True


## 6. TRACE từng bước

Bảng cho `n = 600`:

| p | n trước | n % p == 0? | hành động | n sau |
|---:|---:|:---:|---|---:|
| 2 | 600 | có | 600 // 2 | 300 |
| 2 | 300 | có | 300 // 2 | 150 |
| 2 | 150 | có | 150 // 2 | 75 |
| 2 | 75 | không | sang p = 3 | 75 |
| 3 | 75 | có | 75 // 3 | 25 |
| 3 | 25 | không | sang p = 5 | 25 |
| 5 | 25 | có | 25 // 5 | 5 |
| 5 | 5 | có | 5 // 5 | 1 |
| 5 | 1 | không | kết thúc | 1 |

`n == 1` → `True`.

In [11]:
def trace_is_ugly(n: int) -> bool:
    print(f"Kiểm tra n = {n}")
    if n <= 0:
        print("  n <= 0 -> False ngay")
        return False
    for p in (2, 3, 5):
        while n % p == 0:
            print(f"  p = {p}: {n} % {p} == 0 -> n = {n} // {p} = {n // p}")
            n //= p
        print(f"  p = {p}: {n} % {p} = {n % p} != 0 -> chuyển tiếp")
    print(f"  phần còn lại = {n} -> {'True' if n == 1 else 'False (còn thừa số lạ)'}")
    return n == 1

trace_is_ugly(600)
print()
trace_is_ugly(14)
print()
trace_is_ugly(0)

Kiểm tra n = 600
  p = 2: 600 % 2 == 0 -> n = 600 // 2 = 300
  p = 2: 300 % 2 == 0 -> n = 300 // 2 = 150
  p = 2: 150 % 2 == 0 -> n = 150 // 2 = 75
  p = 2: 75 % 2 = 1 != 0 -> chuyển tiếp
  p = 3: 75 % 3 == 0 -> n = 75 // 3 = 25
  p = 3: 25 % 3 = 1 != 0 -> chuyển tiếp
  p = 5: 25 % 5 == 0 -> n = 25 // 5 = 5
  p = 5: 5 % 5 == 0 -> n = 5 // 5 = 1
  p = 5: 1 % 5 = 1 != 0 -> chuyển tiếp
  phần còn lại = 1 -> True

Kiểm tra n = 14
  p = 2: 14 % 2 == 0 -> n = 14 // 2 = 7
  p = 2: 7 % 2 = 1 != 0 -> chuyển tiếp
  p = 3: 7 % 3 = 1 != 0 -> chuyển tiếp
  p = 5: 7 % 5 = 2 != 0 -> chuyển tiếp
  phần còn lại = 7 -> False (còn thừa số lạ)

Kiểm tra n = 0
  n <= 0 -> False ngay


False

## 7. Code nộp LeetCode

In [12]:
class Solution:
    def isUgly(self, n: int) -> bool:
        if n <= 0:
            return False
        for p in (2, 3, 5):
            while n % p == 0:
                n //= p
        return n == 1

sol = Solution()
print(sol.isUgly(6), sol.isUgly(1), sol.isUgly(14))

True True False


## 8. Vì sao đúng? (invariant)

**Bất biến:** sau mỗi phép `n //= p` (với `p ∈ {2,3,5}`), *"`n` ban đầu là ugly ⇔ `n` hiện tại là ugly"*. Vì ta chỉ bỏ đi một thừa số **được phép** — tập thừa số "lạ" không đổi.

**Kết thúc vòng `p`:** `n` không còn chia hết cho `p`. Sau cả ba vòng, `n` không chia hết cho 2, 3, 5.
- Nếu `n == 1`: không còn thừa số nào → ugly.
- Nếu `n > 1`: theo định lý cơ bản, `n` có ít nhất một thừa số nguyên tố `q`, và `q ∉ {2,3,5}` → không ugly.

**Dừng:** mỗi phép chia làm `n` giảm ít nhất một nửa (với `n > 0`), nên tổng số phép chia `<= log2(n) <= 31`.

## 9. Hai cách sáng tạo khác

### 9.1. Cách 3 — Mẹo bit: gỡ toàn bộ thừa số 2 trong một bước

Với số nguyên dương `n`, biểu thức `n & -n` cho **lũy thừa 2 lớn nhất chia hết `n`** (bit 1 thấp nhất). Ví dụ `n = 24 = 0b11000` → `n & -n = 0b1000 = 8`, và `24 // 8 = 3`.

Lý do (tóm tắt): `-n` trong bù hai là `~n + 1`; phép `+1` lật các bit 0 ở cuối và dừng ở bit 1 thấp nhất, nên `n` và `-n` chỉ cùng có bit đó.

In [13]:
for n in [24, 600, 7, 64]:
    low = n & -n
    print(f"n = {n:>3} = {bin(n):>10} | n & -n = {low:>3} | n // (n & -n) = {n // low}")

def is_ugly_bit(n: int) -> bool:
    if n <= 0:
        return False
    n //= (n & -n)              # gỡ hết thừa số 2 một lần
    for p in (3, 5):
        while n % p == 0:
            n //= p
    return n == 1

print([is_ugly_bit(x) for x in [6, 1, 14, 0, -6, 600]])

n =  24 =    0b11000 | n & -n =   8 | n // (n & -n) = 3
n = 600 = 0b1001011000 | n & -n =   8 | n // (n & -n) = 75
n =   7 =      0b111 | n & -n =   1 | n // (n & -n) = 7
n =  64 =  0b1000000 | n & -n =  64 | n // (n & -n) = 1
[True, True, False, False, False, True]


### 9.2. Cách 4 — Một phép `%`: "bội chung khổng lồ"

Trong giới hạn `n <= 2^31 - 1`: lũy thừa lớn nhất của 2 là `2^30`, của 3 là `3^19 = 1162261467`, của 5 là `5^13 = 1220703125`.

Đặt `M = 2^30 · 3^19 · 5^13`. Mọi ugly number trong giới hạn đều có dạng `2^a 3^b 5^c` với `a<=30, b<=19, c<=13` → **chia hết `M`**. Ngược lại, nếu `n` chia hết `M` thì thừa số của `n` nằm trong thừa số của `M` → chỉ gồm 2, 3, 5.

⇒ `n` ugly ⇔ `n > 0` và `M % n == 0`. Python hỗ trợ số nguyên lớn nên `M` (~1.5·10^27) không tràn.

In [14]:
print("2^30 =", 2**30, "| 2^31 =", 2**31, "(> 2^31 - 1)")
print("3^19 =", 3**19, "| 3^20 =", 3**20)
print("5^13 =", 5**13, "| 5^14 =", 5**14)
M = 2**30 * 3**19 * 5**13
print("M =", M)

def is_ugly_magic(n: int) -> bool:
    return n > 0 and M % n == 0

print([is_ugly_magic(x) for x in [6, 1, 14, 0, -6, 600]])

2^30 = 1073741824 | 2^31 = 2147483648 (> 2^31 - 1)
3^19 = 1162261467 | 3^20 = 3486784401
5^13 = 1220703125 | 5^14 = 6103515625
M = 1523399350026240000000000000
[True, True, False, False, False, True]


**Lưu ý:** cách 4 phụ thuộc vào giới hạn đề (`n < 2^31`). Nếu đề đổi giới hạn, phải tính lại `M`. Trong phỏng vấn nên trình bày cách 2 trước, cách 4 là "điểm cộng".

## 10. Độ phức tạp và bảng so sánh

Gọi `n` là giá trị input.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Phân tích đầy đủ | thử chia tới `√n` | `O(√n)` | `O(log n)` (list thừa số) | dư thừa thông tin |
| 2. Gỡ 2, 3, 5 (chính) | chia tới khi hết | `O(log n)` | `O(1)` | chuẩn phỏng vấn |
| 3. Bit `n & -n` | gỡ thừa số 2 một bước | `O(log n)` | `O(1)` | khéo, ít phép chia hơn |
| 4. `M % n == 0` | bội chung khổng lồ | `O(1)` theo đề (số lớn cố định) | `O(1)` | phụ thuộc giới hạn |

Số phép chia ở cách 2 `<= a + b + c + 3 <= log2(n) + 3`.

## 11. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<32} PASS")

sol = Solution()
check("mẫu 1: n = 6", sol.isUgly(6), True)
check("mẫu 2: n = 1", sol.isUgly(1), True)
check("mẫu 3: n = 14", sol.isUgly(14), False)
check("biên: n = 0 (không lặp vô hạn)", sol.isUgly(0), False)
check("âm: n = -6", sol.isUgly(-6), False)
check("min int", sol.isUgly(-2**31), False)
check("số nguyên tố lớn 2^31-1", sol.isUgly(2**31 - 1), False)
check("2^30", sol.isUgly(2**30), True)
check("3^19 (lũy thừa 3 lớn nhất)", sol.isUgly(3**19), True)
check("số nguyên tố 7", sol.isUgly(7), False)
check("2*3*5*7 = 210", sol.isUgly(210), False)

mẫu 1: n = 6                     PASS
mẫu 2: n = 1                     PASS
mẫu 3: n = 14                    PASS
biên: n = 0 (không lặp vô hạn)   PASS
âm: n = -6                       PASS
min int                          PASS
số nguyên tố lớn 2^31-1          PASS
2^30                             PASS
3^19 (lũy thừa 3 lớn nhất)       PASS
số nguyên tố 7                   PASS
2*3*5*7 = 210                    PASS


In [16]:
# Đối chiếu 4 cách trên miền nhỏ và trên mọi số dạng 2^a 3^b 5^c trong giới hạn
import random
funcs = [is_ugly_factorize, is_ugly_divide, is_ugly_bit, is_ugly_magic, Solution().isUgly]
for x in range(-50, 20001):
    assert len({f(x) for f in funcs}) == 1, x

uglies = sorted(2**a * 3**b * 5**c for a in range(31) for b in range(20) for c in range(14)
                if 2**a * 3**b * 5**c <= 2**31 - 1)
print("Số ugly number trong [1, 2^31-1]:", len(uglies), "| 10 số đầu:", uglies[:10])
assert all(is_ugly_divide(u) and is_ugly_magic(u) and is_ugly_bit(u) for u in uglies)
rng = random.Random(263)
for _ in range(3000):
    x = rng.choice(uglies) * rng.choice([1, 7, 11, 13, 49])
    if x <= 2**31 - 1:
        assert is_ugly_divide(x) == is_ugly_magic(x) == is_ugly_bit(x) == (x in set(uglies))
print("Các cách khớp nhau trên mọi test")

Số ugly number trong [1, 2^31-1]: 1691 | 10 số đầu: [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]


Các cách khớp nhau trên mọi test


## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên chặn `n == 0` → vòng lặp vô hạn

```python
def wrong(n):
    for p in (2, 3, 5):
        while n % p == 0:   # n = 0: 0 % 2 == 0 mãi mãi
            n //= p
    return n == 1
```

Không chạy cell này (sẽ treo). Ta mô phỏng có giới hạn để thấy vấn đề:

In [17]:
def wrong_no_zero_guard_limited(n, max_steps=10):
    steps = 0
    for p in (2, 3, 5):
        while n % p == 0:
            n //= p
            steps += 1
            if steps >= max_steps:
                return f"đã {steps} bước mà chưa dừng (n = {n}) -> vô hạn"
    return n == 1

print(wrong_no_zero_guard_limited(0))
print(wrong_no_zero_guard_limited(6))

đã 10 bước mà chưa dừng (n = 0) -> vô hạn
True


**Sửa:** `if n <= 0: return False` ở đầu hàm.

### Lỗi 2: Dùng `if` thay vì `while` — chỉ gỡ một lần

In [18]:
def wrong_if(n):
    if n <= 0:
        return False
    for p in (2, 3, 5):
        if n % p == 0:          # SAI: 8 = 2*2*2 chỉ gỡ được một 2
            n //= p
    return n == 1

print("wrong_if(8) =", wrong_if(8), "| đúng =", Solution().isUgly(8))

wrong_if(8) = False | đúng = True


### Lỗi 3: Dùng `/` thay `//`

`n / 2` ra số thực. `6 / 2 / 3 = 1.0`, và `1.0 == 1` là `True` nên nhiều ca vẫn "may mắn" đúng, nhưng với số lớn, số thực mất chính xác và `%` trên float dễ sai. Luôn dùng `//` cho số nguyên.

In [19]:
print("6 / 2 =", 6 / 2, type(6 / 2).__name__, "| 6 // 2 =", 6 // 2, type(6 // 2).__name__)
big = 2**60 * 3 + 1
print("Số lớn chia /:", big / 3, "| chia //:", big // 3, "(float làm tròn mất chữ số cuối)")

6 / 2 = 3.0 float | 6 // 2 = 3 int
Số lớn chia /: 1.152921504606847e+18 | chia //: 1152921504606846976 (float làm tròn mất chữ số cuối)


### Lỗi 4: Nghĩ `1` không phải ugly

Đề nói rõ `1` là ugly (không có thừa số nguyên tố). Hàm đúng cho `True` vì không gỡ được gì và `n == 1`. Đừng thêm `if n == 1: return False`.

### Lỗi 5: Kiểm tra "chia hết cho 2, 3 hoặc 5" thay vì "**chỉ** gồm 2, 3, 5"

`14` chia hết cho 2 nhưng vẫn không ugly. Phải gỡ hết rồi xem phần còn lại.

In [20]:
def wrong_any(n):
    return n > 0 and (n % 2 == 0 or n % 3 == 0 or n % 5 == 0)

print("wrong_any(14) =", wrong_any(14), "| đúng =", Solution().isUgly(14))
print("wrong_any(1) =", wrong_any(1), "| đúng =", Solution().isUgly(1))

wrong_any(14) = True | đúng = False
wrong_any(1) = False | đúng = True


## 13. Mẹo nhận dạng pattern

- **"Chỉ gồm các thừa số nguyên tố trong tập S"** → gỡ hết các thừa số thuộc S, kiểm tra phần còn lại `== 1`.
- **"Có phải lũy thừa của k"** (bài 231 Power of Two, 326 Power of Three, 342 Power of Four) → cùng khung: `while n % k == 0: n //= k; return n == 1`, hoặc mẹo "bội khổng lồ" `3^19 % n == 0`.
- Thấy `while n % p == 0` → **luôn** hỏi "n có thể bằng 0 không?".
- Cần **sinh** dãy ugly number (bài 264 Ugly Number II) → không kiểm tra từng số, mà dùng 3 con trỏ / heap.

## 14. Biến thể

### 14.1. Tổng quát: chỉ gồm thừa số trong tập `primes` bất kỳ

In [21]:
def only_factors(n: int, primes: tuple[int, ...]) -> bool:
    if n <= 0:
        return False
    for p in primes:
        while n % p == 0:
            n //= p
    return n == 1

print(only_factors(2**5 * 7**2, (2, 7)), only_factors(2**5 * 7**2, (2, 3)), only_factors(1, ()))

True False True


### 14.2. Ugly Number II (LeetCode 264): số ugly thứ `k`

Brute force kiểm tra từng số thì quá chậm khi `k` lớn. Ý tưởng 3 con trỏ: mỗi ugly number tiếp theo là `min(u[i2]*2, u[i3]*3, u[i5]*5)`.

In [22]:
def nth_ugly(k: int) -> int:
    u = [1]
    i2 = i3 = i5 = 0
    while len(u) < k:
        nxt = min(u[i2] * 2, u[i3] * 3, u[i5] * 5)
        u.append(nxt)
        if nxt == u[i2] * 2: i2 += 1
        if nxt == u[i3] * 3: i3 += 1      # không dùng elif: 6 = 2*3 = 3*2 phải tăng cả hai
        if nxt == u[i5] * 5: i5 += 1
    return u[-1]

print([nth_ugly(k) for k in range(1, 16)])
assert [nth_ugly(k) for k in range(1, 200)] == [x for x in range(1, 10**6) if is_ugly_divide(x)][:199]
print("nth_ugly khớp brute force cho 199 số đầu")

[1, 2, 3, 4, 5, 6, 8, 9, 10, 12, 15, 16, 18, 20, 24]


nth_ugly khớp brute force cho 199 số đầu


### 14.3. Power of Three (LeetCode 326) — cùng khuôn

In [23]:
def is_power_of_three(n: int) -> bool:
    return n > 0 and 3**19 % n == 0

print([x for x in range(1, 300) if is_power_of_three(x)])

[1, 3, 9, 27, 81, 243]


## 15. Bài tập (tự làm — cell chỉ định nghĩa, không gọi)

**Bài 1 — Tính tay:** Với `n = 1500`, mỗi bước gỡ thừa số cho `n` bằng bao nhiêu? Phần còn lại là gì? Ghi danh sách các giá trị `n` sau mỗi phép chia.

In [24]:
# TODO: điền list các giá trị n sau MỖI phép chia (theo thứ tự p = 2, 3, 5)
predict_1500_steps = None
# assert predict_1500_steps == <xem đáp án>

**Bài 2 — Điền chỗ trống:** hoàn thiện `exercise_is_ugly`.

In [25]:
def exercise_is_ugly(n: int) -> bool:
    # TODO 1: chặn n <= 0
    # TODO 2: với mỗi p trong (2, 3, 5), gỡ hết p
    # TODO 3: trả về phần còn lại có bằng 1 không
    pass

# assert [exercise_is_ugly(x) for x in [6, 1, 14, 0, -6]] == [True, True, False, False, False]

**Bài 3 — Đếm số mũ:** trả về tuple `(a, b, c)` sao cho `n = 2^a 3^b 5^c`, hoặc `None` nếu `n` không ugly.

In [26]:
def exercise_exponents(n: int):
    # TODO
    pass

# assert exercise_exponents(600) == (3, 1, 2) and exercise_exponents(14) is None and exercise_exponents(1) == (0, 0, 0)

**Bài 4 — Power of Two:** viết `exercise_power_of_two(n)` bằng **hai** cách: vòng chia và mẹo bit (`n & (n - 1) == 0`). Trả về tuple kết quả 2 cách.

In [27]:
def exercise_power_of_two(n: int) -> tuple[bool, bool]:
    # TODO
    pass

# assert exercise_power_of_two(16) == (True, True) and exercise_power_of_two(0) == (False, False) and exercise_power_of_two(6) == (False, False)

**Bài 5 — Đếm ugly number trong `[1, N]`** (`N <= 10^5`).

In [28]:
def exercise_count_ugly(N: int) -> int:
    # TODO
    pass

# assert exercise_count_ugly(10) == 9 and exercise_count_ugly(1) == 1

**Bài 6 — Mini challenge:** số ugly thứ `k` (Ugly Number II) **không** nhìn mục 14.2: thử dùng `heapq` (min-heap) + `set` chống trùng.

In [29]:
def exercise_nth_ugly_heap(k: int) -> int:
    # TODO: import heapq; heap = [1]; seen = {1}; pop k lần
    pass

# assert [exercise_nth_ugly_heap(k) for k in range(1, 11)] == [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `1500 = 2^2 · 3 · 5^3` | gỡ hai lần 2, một lần 3, ba lần 5 | bắt đầu: 750, 375, ... (tự làm tiếp) |
| 2 | khung 3 đoạn ở mục 5 | `for p in (2,3,5): while ...` | `return n == 1` |
| 3 | đếm trong vòng `while` | dùng dict `{2:0,3:0,5:0}` | trả `None` nếu còn lại `!= 1` |
| 4 | lũy thừa 2 có đúng 1 bit 1 | `n & (n-1)` xóa bit 1 thấp nhất | nhớ `n > 0` |
| 5 | gọi hàm kiểm tra cho từng số | `sum(1 for x in ... if ...)` | `range(1, N + 1)` |
| 6 | lấy số nhỏ nhất, đẩy `x*2, x*3, x*5` | dùng `seen` để không đẩy trùng | `heappop` k lần, lần thứ k là đáp án |

## 17. Đáp án tham khảo

In [30]:
answer_1500_steps = [750, 375, 125, 25, 5, 1]

def answer_is_ugly(n: int) -> bool:
    if n <= 0:
        return False
    for p in (2, 3, 5):
        while n % p == 0:
            n //= p
    return n == 1

def answer_exponents(n: int):
    if n <= 0:
        return None
    exps = {2: 0, 3: 0, 5: 0}
    for p in (2, 3, 5):
        while n % p == 0:
            n //= p
            exps[p] += 1
    return (exps[2], exps[3], exps[5]) if n == 1 else None

def answer_power_of_two(n: int) -> tuple[bool, bool]:
    a = n
    if a > 0:
        while a % 2 == 0:
            a //= 2
    loop_way = n > 0 and a == 1
    bit_way = n > 0 and (n & (n - 1)) == 0
    return (loop_way, bit_way)

def answer_count_ugly(N: int) -> int:
    return sum(1 for x in range(1, N + 1) if answer_is_ugly(x))

import heapq
def answer_nth_ugly_heap(k: int) -> int:
    heap, seen = [1], {1}
    for _ in range(k):
        x = heapq.heappop(heap)
        for p in (2, 3, 5):
            if x * p not in seen:
                seen.add(x * p)
                heapq.heappush(heap, x * p)
    return x

# kiểm tra bài 1 bằng code
n, steps = 1500, []
for p in (2, 3, 5):
    while n % p == 0:
        n //= p
        steps.append(n)
assert steps == answer_1500_steps
assert [answer_is_ugly(x) for x in [6, 1, 14, 0, -6]] == [True, True, False, False, False]
assert answer_exponents(600) == (3, 1, 2) and answer_exponents(14) is None and answer_exponents(1) == (0, 0, 0)
assert answer_power_of_two(16) == (True, True) and answer_power_of_two(0) == (False, False) and answer_power_of_two(6) == (False, False)
assert all(answer_power_of_two(x)[0] == answer_power_of_two(x)[1] for x in range(-5, 5000))
assert answer_count_ugly(10) == 9 and answer_count_ugly(1) == 1
assert [answer_nth_ugly_heap(k) for k in range(1, 11)] == [1, 2, 3, 4, 5, 6, 8, 9, 10, 12]
assert all(answer_nth_ugly_heap(k) == nth_ugly(k) for k in range(1, 300))
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def isUgly(self, n: int) -> bool:
        if n <= 0:
            return False
        for p in (2, 3, 5):
            while n % p == 0:
                n //= p
        return n == 1
```

**Câu thần chú:** *"Chặn số không dương → gỡ sạch 2, 3, 5 → còn lại phải là 1."*

**Checklist:**
- [ ] `n <= 0` trả `False` **trước** vòng lặp (tránh treo với 0).
- [ ] Dùng `while` (gỡ hết), không phải `if` (gỡ một lần).
- [ ] Dùng `//`, không dùng `/`.
- [ ] `n = 1` trả `True`.
- [ ] Tên method `isUgly`, trả `bool`.